# Falcon Eye: Data Ingestion Pipeline

<h3>Setup and Dynamic Link</h3>

In [140]:
import pandas as pd


df_meast = pd.read_csv("../../data/streaming/Middle-East.csv")
df_africa = pd.read_csv("../../data/streaming/Africa.csv")


<h3>Merge Data</h3>

In [141]:

combined_df = pd.concat([df_africa, df_meast], ignore_index=True)

<h3>Filtering out non-MENA countries</h3>

In [142]:

mena_countries = [
    "Algeria", "Bahrain", "Egypt", "Iran", "Iraq", "Israel", "Jordan", 
    "Kuwait", "Lebanon", "Libya", "Morocco", "Oman", "Palestine", 
    "Qatar", "Saudi Arabia", "Syria", "Tunisia", "United Arab Emirates", "Yemen"
]

# Filter the dataframe
df_mena = combined_df[combined_df['COUNTRY'].isin(mena_countries)]

print(df_mena.head())

         WEEK           REGION  COUNTRY ADMIN1 EVENT_TYPE SUB_EVENT_TYPE  \
0  10/23/2004  Northern Africa  Algeria  Adrar    Battles    Armed clash   
1  04/23/2005  Northern Africa  Algeria  Adrar    Battles    Armed clash   
2  06/25/2005  Northern Africa  Algeria  Adrar    Battles    Armed clash   
3  12/13/2008  Northern Africa  Algeria  Adrar    Battles    Armed clash   
4  04/18/2009  Northern Africa  Algeria  Adrar    Battles    Armed clash   

   EVENTS  FATALITIES  POPULATION_EXPOSURE       DISORDER_TYPE    ID  \
0       1           2                  NaN  Political violence  47.0   
1       1           0                  NaN  Political violence  47.0   
2       1          14                  NaN  Political violence  47.0   
3       1           3                  NaN  Political violence  47.0   
4       1           2                  NaN  Political violence  47.0   

   CENTROID_LATITUDE  CENTROID_LONGITUDE  
0            26.4839              -1.388  
1            26.4839    

<h3>Filtering out unwanted event types</h3>

In [143]:

df_filtered = df_mena[df_mena['EVENT_TYPE'] != 'Sexual violence']

print(df_filtered['EVENT_TYPE'].unique())

<ArrowStringArray>
[                   'Battles', 'Explosions/Remote violence',
                   'Protests',                      'Riots',
     'Strategic developments', 'Violence against civilians']
Length: 6, dtype: str


<h3>Formatting, Date Conversion, and RowKey Generation</h3>

In [144]:
month_map = {
    'January': '01', 'February': '02', 'March': '03', 'April': '04',
    'May': '05', 'June': '06', 'July': '07', 'August': '08',
    'September': '09', 'October': '10', 'November': '11', 'December': '12'
}

def format_date_for_hbase(date_str):
    try:
        parts = str(date_str).split('/')
        day = parts[1]
        month =  parts[0]
        year = parts[2]

        return f"{year}-{month}-{day}"
    except Exception:
        return date_str

df_filtered['WEEK'] = df_filtered['WEEK'].apply(format_date_for_hbase)



In [145]:
df_filtered['row_key'] = df_filtered['COUNTRY'] + "#" + df_filtered['WEEK'] + "#" + df_filtered['ID'].astype(str)


cols = ['row_key'] + [c for c in df_filtered.columns if c != 'row_key']
df_filtered = df_filtered[cols]


<h3>Filtering out the old data and keep only the latest rows</h3>

In [146]:
# Get the latest event from our hbase table
#  scan 'events', {COLUMNS => 'cf:week', LIMIT => 2,REVERSED => true }
# ==> 2026-04-11 

In [147]:
df_filtered['WEEK'] = pd.to_datetime(df_filtered['WEEK'])

cutoff_date = '2026-04-11'

df_new_only = df_filtered[df_filtered['WEEK'] > cutoff_date]

# 4. Verification: Print the range of dates remaining
print(f"Oldest date in new set: {df_new_only['WEEK'].min()}")
print(f"Total rows remaining: {len(df_new_only)}")

df_new_only.to_csv('../../data/streaming/stream-mena.csv', index=False, header=False,)


Oldest date in new set: 2026-04-18 00:00:00
Total rows remaining: 870


<h3>Prepre data for flume</h3>